In [1]:
# Load analysis libraries and the reviewed player-ID matcher.
import pandas as pd
import duckdb
import numpy as np
import sys
from pathlib import Path

sys.path.insert(0, str((Path.cwd().parent / "scripts").resolve()))
from player_id_matcher import match_transaction_players
from typing import Any, Mapping

In [2]:
# Load transactions and smoke-test the matcher on one row.
transactions = pd.read_csv("../data/interim/transactions_with_WL.csv")

result = match_transaction_players(transactions.iloc[0], include_details=True)
result

{'Acquired': {},
 'Relinquished': {'Mack Calvin': {'player_id': 76336,
   'match_status': 'exact_match',
   'box_score_player_name': 'Mack Calvin',
   'most_recent_season': '1976-77',
   'model_value': None}}}

In [3]:
team_Games = duckdb.read_parquet("../data/interim/filtered_Team_Stats.parquet").df()

In [4]:
box_Score = duckdb.read_parquet("../data/interim/filtered_Player_Stats.parquet").df()

In [5]:
# Normalize dates before constructing season segments.
box_Score["game_Date"] = pd.to_datetime(box_Score["game_Date"]).dt.normalize()
transactions["Date"] = pd.to_datetime(transactions["Date"]).dt.normalize()

In [6]:
# Offseason transactions use the completed previous season for season-level features.
def previous_season_from_date(date):
    date = pd.Timestamp(date)
    return f"{date.year - 1}-{str(date.year)[-2:]}"


def upcoming_season_from_date(date):
    date = pd.Timestamp(date)
    return f"{date.year}-{str(date.year + 1)[-2:]}"


transactions["season_segment"] = np.where(transactions["Season"].astype(str).eq("Offseason"), "offseason", "regular_season")
transactions["comparison_season"] = np.where(
    transactions["season_segment"].eq("offseason"), transactions["Date"].map(upcoming_season_from_date), transactions["Season"]
)
transactions["feature_season"] = np.where(
    transactions["season_segment"].eq("offseason"), transactions["Date"].map(previous_season_from_date), transactions["Season"]
)

In [7]:
print("Invalid box-score dates:", box_Score["game_Date"].isna().sum())
print("Invalid transaction dates:", transactions["Date"].isna().sum())

Invalid box-score dates: 0
Invalid transaction dates: 0


In [8]:
# Define player box-score fields that must be numeric.
numeric_columns = [
    "personId",
    "gameId",
    "playerteamId",
    "opponentteamId",
    "win",
    "home",
    "numMinutes",
    "points",
    "assists",
    "blocks",
    "steals",
    "fieldGoalsAttempted",
    "fieldGoalsMade",
    "fieldGoalsPercentage",
    "threePointersAttempted",
    "threePointersMade",
    "threePointersPercentage",
    "freeThrowsAttempted",
    "freeThrowsMade",
    "freeThrowsPercentage",
    "reboundsDefensive",
    "reboundsOffensive",
    "reboundsTotal",
    "foulsPersonal",
    "turnovers",
    "plusMinusPoints",
]

In [9]:
for column in numeric_columns:
    box_Score[column] = pd.to_numeric(box_Score[column])

In [10]:
# Flag actual appearances and calculate minutes played.
box_Score["minutesPlayed"] = box_Score["numMinutes"].fillna(0.0)

box_Score["appeared"] = box_Score["minutesPlayed"] > 0

In [11]:
box_Score["started"] = box_Score["startingPosition"].fillna("").astype(str).str.strip().ne("") & box_Score["appeared"]

In [12]:
# Assign player games to NBA seasons.
season_start_year = box_Score["game_Date"].dt.year - (box_Score["game_Date"].dt.month < 7).astype(int)

box_Score["Season"] = season_start_year.astype(str) + "-" + (season_start_year + 1).astype(str).str[-2:]

In [13]:
# Derive two-point scoring and possession inputs.
box_Score["twoPointersMade"] = box_Score["fieldGoalsMade"] - box_Score["threePointersMade"]

box_Score["twoPointersAttempted"] = box_Score["fieldGoalsAttempted"] - box_Score["threePointersAttempted"]

In [14]:
# Check the player-game grain for duplicate rows.
duplicate_mask = box_Score.duplicated(subset=["personId", "gameId", "playerteamId"], keep=False)

box_Score.loc[duplicate_mask, ["personId", "gameId", "playerteamId", "game_Date"]].head(20)

,personId,gameId,playerteamId,game_Date


In [15]:
team_Games["game_Date"] = pd.to_datetime(team_Games["game_Date"]).dt.normalize()

In [16]:
team_Games["game_Date"].isna().sum()

np.int64(0)

In [17]:
# Assign team games to NBA seasons.
season_start_year = team_Games["game_Date"].dt.year - (team_Games["game_Date"].dt.month < 7).astype(int)

team_Games["Season"] = season_start_year.astype(str) + "-" + (season_start_year + 1).astype(str).str[-2:]

In [18]:
# Define team context fields that must be numeric.
team_numeric_columns = [
    "gameId",
    "teamId",
    "opponentTeamId",
    "home",
    "win",
    "teamScore",
    "opponentScore",
    "assists",
    "blocks",
    "steals",
    "fieldGoalsAttempted",
    "fieldGoalsMade",
    "threePointersAttempted",
    "threePointersMade",
    "freeThrowsAttempted",
    "freeThrowsMade",
    "reboundsDefensive",
    "reboundsOffensive",
    "reboundsTotal",
    "reboundsTeam",
    "foulsPersonal",
    "turnovers",
    "turnoversTeam",
    "plusMinusPoints",
    "numMinutes",
]

In [19]:
for column in team_numeric_columns:
    team_Games[column] = pd.to_numeric(team_Games[column])

In [20]:
# Exclude seasons that do not meet the project coverage standard.
excluded_seasons = ["1976-77", "1977-78", "1978-79", "1979-80", "1980-81", "1981-82", "1982-83", "1983-84", "1984-85", "2000-01"]

box_Score = box_Score.loc[~box_Score["Season"].isin(excluded_seasons)].copy().reset_index(drop=True)

team_Games = team_Games.loc[~team_Games["Season"].isin(excluded_seasons)].copy().reset_index(drop=True)

transactions = transactions.loc[~transactions["feature_season"].isin(excluded_seasons)].copy().reset_index(drop=True)

In [21]:
# Combine team turnover components used by possession estimates.
team_Games["total_team_turnovers"] = team_Games["turnovers"].fillna(0) + team_Games["turnoversTeam"].fillna(0)

In [22]:
team_Games = team_Games.rename(
    columns={
        "teamId": "playerteamId",
        "opponentTeamId": "opponentteamId",
        "teamScore": "team_points",
        "opponentScore": "opp_points",
        "fieldGoalsAttempted": "team_fga",
        "fieldGoalsMade": "team_fgm",
        "threePointersAttempted": "team_fg3a",
        "threePointersMade": "team_fg3m",
        "freeThrowsAttempted": "team_fta",
        "freeThrowsMade": "team_ftm",
        "reboundsDefensive": "team_drb",
        "reboundsOffensive": "team_orb",
        "reboundsTotal": "team_trb",
        "total_team_turnovers": "team_tov",
    }
)

In [23]:
# Build opponent totals for each team-game row.
opponent_totals = team_Games[
    [
        "gameId",
        "playerteamId",
        "team_fga",
        "team_fgm",
        "team_fg3a",
        "team_fg3m",
        "team_fta",
        "team_ftm",
        "team_drb",
        "team_orb",
        "team_trb",
        "team_tov",
    ]
].rename(
    columns={
        "playerteamId": "opponentteamId",
        "team_fga": "opp_fga",
        "team_fgm": "opp_fgm",
        "team_fg3a": "opp_fg3a",
        "team_fg3m": "opp_fg3m",
        "team_fta": "opp_fta",
        "team_ftm": "opp_ftm",
        "team_drb": "opp_drb",
        "team_orb": "opp_orb",
        "team_trb": "opp_trb",
        "team_tov": "opp_tov",
    }
)

In [24]:
# Attach opponent context to the team records.
team_Games = team_Games.merge(opponent_totals, on=["gameId", "opponentteamId"], how="left", validate="one_to_one")

In [25]:
team_orb_denominator = team_Games["team_orb"] + team_Games["opp_drb"]

opp_orb_denominator = team_Games["opp_orb"] + team_Games["team_drb"]

team_orb_share = team_Games["team_orb"] / team_orb_denominator.replace(0, np.nan)

opp_orb_share = team_Games["opp_orb"] / opp_orb_denominator.replace(0, np.nan)

In [26]:
# Estimate team and game possessions.
team_Games["team_possession_estimate"] = (
    team_Games["team_fga"]
    + 0.4 * team_Games["team_fta"]
    - 1.07 * team_orb_share * (team_Games["team_fga"] - team_Games["team_fgm"])
    + team_Games["team_tov"]
)

team_Games["opp_possession_estimate"] = (
    team_Games["opp_fga"]
    + 0.4 * team_Games["opp_fta"]
    - 1.07 * opp_orb_share * (team_Games["opp_fga"] - team_Games["opp_fgm"])
    + team_Games["opp_tov"]
)

team_Games["estimated_possessions"] = (team_Games["team_possession_estimate"] + team_Games["opp_possession_estimate"]) / 2

In [27]:
team_Games["pace"] = 48 * team_Games["estimated_possessions"] / team_Games["numMinutes"].replace(0, np.nan)

In [28]:
team_Games["point_differential"] = team_Games["team_points"] - team_Games["opp_points"]

team_Games["team_offensive_rating"] = 100 * team_Games["team_points"] / team_Games["estimated_possessions"].replace(0, np.nan)

team_Games["team_defensive_rating"] = 100 * team_Games["opp_points"] / team_Games["estimated_possessions"].replace(0, np.nan)

team_Games["team_net_rating"] = team_Games["team_offensive_rating"] - team_Games["team_defensive_rating"]

In [29]:
plus_minus_difference = team_Games["point_differential"] - team_Games["plusMinusPoints"]

plus_minus_difference.value_counts(dropna=False).head(20)

 0.0     75432
-1.0        12
 1.0        12
-2.0        12
 2.0        12
-3.0         5
 3.0         5
 10.0        5
-10.0        5
 20.0        2
-20.0        2
-6.0         1
 6.0         1
Name: count, dtype: int64

In [30]:
print("Player box-score date range:", box_Score["game_Date"].min(), "to", box_Score["game_Date"].max())

print("Team box-score date range:", team_Games["game_Date"].min(), "to", team_Games["game_Date"].max())

Player box-score date range: 1985-10-25 00:00:00 to 2019-04-10 00:00:00
Team box-score date range: 1985-10-25 00:00:00 to 2019-04-10 00:00:00


In [31]:
player_game_ids = set(box_Score["gameId"].dropna())

team_game_ids = set(team_Games["gameId"].dropna())

missing_team_games = player_game_ids - team_game_ids

len(missing_team_games)

0

In [32]:
# Remove the unusable 2000-01 season from both stat tables.
box_Score = box_Score.loc[box_Score["Season"] != "2000-01"].copy().reset_index(drop=True)

team_Games = team_Games.loc[team_Games["Season"] != "2000-01"].copy().reset_index(drop=True)

In [33]:
player_game_ids = set(box_Score["gameId"].dropna())

team_game_ids = set(team_Games["gameId"].dropna())

missing_team_games = player_game_ids - team_game_ids

len(missing_team_games)

0

In [34]:
# Remove transactions whose feature season is 2000-01.
transactions = transactions.loc[transactions["feature_season"] != "2000-01"].copy().reset_index(drop=True)

In [35]:
team_Games.duplicated(subset=["gameId", "game_Date", "Season", "playerteamId", "opponentteamId"], keep=False).sum()

np.int64(0)

In [36]:
team_Games["team_minutes"] = team_Games["numMinutes"]

team_Games["game_minutes"] = team_Games["numMinutes"] / 5

In [37]:
# Select the team context required for player-rate calculations.
team_context_columns = [
    "gameId",
    "game_Date",
    "Season",
    "playerteamId",
    "opponentteamId",
    "team_minutes",
    "game_minutes",
    "team_points",
    "opp_points",
    "team_fga",
    "team_fgm",
    "team_fg3a",
    "team_fg3m",
    "team_fta",
    "team_ftm",
    "team_drb",
    "team_orb",
    "team_trb",
    "team_tov",
    "opp_fga",
    "opp_fgm",
    "opp_fg3a",
    "opp_fg3m",
    "opp_fta",
    "opp_ftm",
    "opp_drb",
    "opp_orb",
    "opp_trb",
    "opp_tov",
    "estimated_possessions",
    "pace",
    "point_differential",
    "team_offensive_rating",
    "team_defensive_rating",
    "team_net_rating",
]

In [38]:
missing_context_columns = [column for column in team_context_columns if column not in team_Games.columns]

missing_context_columns

[]

In [39]:
# Attach team and opponent context to every player box score.
enriched_Box_Scores = box_Score.merge(
    team_Games[team_context_columns],
    on=["gameId", "game_Date", "Season", "playerteamId", "opponentteamId"],
    how="left",
    validate="many_to_one",
)

In [40]:
print("Original player rows:", len(box_Score))

print("Enriched player rows:", len(enriched_Box_Scores))

Original player rows: 876532
Enriched player rows: 876532


In [41]:
# Estimate player possessions from minutes and game pace.
enriched_Box_Scores["estimatedPlayerPossessions"] = (
    enriched_Box_Scores["estimated_possessions"]
    * enriched_Box_Scores["minutesPlayed"]
    / enriched_Box_Scores["game_minutes"].replace(0, np.nan)
)

In [42]:
invalid_player_possessions = enriched_Box_Scores.loc[
    enriched_Box_Scores["appeared"]
    & (enriched_Box_Scores["estimatedPlayerPossessions"].isna() | (enriched_Box_Scores["estimatedPlayerPossessions"] <= 0))
]

len(invalid_player_possessions)

64

In [43]:
appeared_rows = enriched_Box_Scores.loc[enriched_Box_Scores["appeared"]].copy()

print("Player appearances:", len(appeared_rows))

print("Missing estimated player possessions:", appeared_rows["estimatedPlayerPossessions"].isna().sum())

print("Zero or negative estimated player possessions:", (appeared_rows["estimatedPlayerPossessions"] <= 0).sum())

Player appearances: 767789
Missing estimated player possessions: 64
Zero or negative estimated player possessions: 0


In [44]:
possession_columns = [
    "personId",
    "gameId",
    "game_Date",
    "Season",
    "playerteamId",
    "minutesPlayed",
    "team_minutes",
    "game_minutes",
    "estimated_possessions",
    "estimatedPlayerPossessions",
]

invalid_player_possessions[possession_columns]

invalid_player_possessions[possession_columns].isna().sum()

personId                       0
gameId                         0
game_Date                      0
Season                         0
playerteamId                  64
minutesPlayed                  0
team_minutes                  64
game_minutes                  64
estimated_possessions         64
estimatedPlayerPossessions    64
dtype: int64

In [45]:
invalid_game_ids = invalid_player_possessions["gameId"].dropna().unique()

print("Invalid player rows:", len(invalid_player_possessions))

print("Unique affected games:", len(invalid_game_ids))

Invalid player rows: 64
Unique affected games: 4


In [46]:
possession_input_columns = [
    "gameId",
    "game_Date",
    "Season",
    "playerteamId",
    "opponentteamId",
    "team_fga",
    "team_fgm",
    "team_fta",
    "team_orb",
    "team_drb",
    "team_tov",
    "opp_fga",
    "opp_fgm",
    "opp_fta",
    "opp_orb",
    "opp_drb",
    "opp_tov",
    "team_possession_estimate",
    "opp_possession_estimate",
    "estimated_possessions",
]

affected_team_games = team_Games.loc[team_Games["gameId"].isin(invalid_game_ids), possession_input_columns].copy()

affected_team_games[possession_input_columns].isna().sum().sort_values(ascending=False)

gameId                      0
game_Date                   0
Season                      0
playerteamId                0
opponentteamId              0
team_fga                    0
team_fgm                    0
team_fta                    0
team_orb                    0
team_drb                    0
team_tov                    0
opp_fga                     0
opp_fgm                     0
opp_fta                     0
opp_orb                     0
opp_drb                     0
opp_tov                     0
team_possession_estimate    0
opp_possession_estimate     0
estimated_possessions       0
dtype: int64

In [47]:
affected_team_games[["gameId", "Season"]].drop_duplicates()["Season"].value_counts().sort_index()

Season
1988-89    1
1992-93    1
1994-95    1
1999-00    1
Name: count, dtype: int64

In [48]:
all_games_by_season = team_Games[["gameId", "Season"]].drop_duplicates().groupby("Season").size().rename("total_games")

affected_games_by_season = affected_team_games[["gameId", "Season"]].drop_duplicates().groupby("Season").size().rename("affected_games")

possession_coverage_by_season = pd.concat([all_games_by_season, affected_games_by_season], axis=1).fillna(0)

possession_coverage_by_season["affected_percentage"] = (
    100 * possession_coverage_by_season["affected_games"] / possession_coverage_by_season["total_games"]
)

possession_coverage_by_season.loc[possession_coverage_by_season["affected_games"] > 0]

,total_games,affected_games,affected_percentage
Season,,,
1988-89,1025,1.0,0.097561
1992-93,1107,1.0,0.090334
1994-95,1107,1.0,0.090334
1999-00,1189,1.0,0.084104


In [49]:
# Calculate game score for every player appearance.
enriched_Box_Scores["gameScore"] = (
    enriched_Box_Scores["points"]
    + 0.4 * enriched_Box_Scores["fieldGoalsMade"]
    - 0.7 * enriched_Box_Scores["fieldGoalsAttempted"]
    - 0.4 * (enriched_Box_Scores["freeThrowsAttempted"] - enriched_Box_Scores["freeThrowsMade"])
    + 0.7 * enriched_Box_Scores["reboundsOffensive"]
    + 0.3 * enriched_Box_Scores["reboundsDefensive"]
    + enriched_Box_Scores["steals"]
    + 0.7 * enriched_Box_Scores["assists"]
    + 0.7 * enriched_Box_Scores["blocks"]
    - 0.4 * enriched_Box_Scores["foulsPersonal"]
    - enriched_Box_Scores["turnovers"]
)

In [50]:
# Map raw counting stats to reusable feature names.
counting_stat_columns = {
    "points": "points",
    "assists": "assists",
    "turnovers": "turnovers",
    "offensive_rebounds": "reboundsOffensive",
    "defensive_rebounds": "reboundsDefensive",
    "total_rebounds": "reboundsTotal",
    "steals": "steals",
    "blocks": "blocks",
    "personal_fouls": "foulsPersonal",
    "field_goals_made": "fieldGoalsMade",
    "field_goals_attempted": "fieldGoalsAttempted",
    "three_pointers_made": "threePointersMade",
    "three_pointers_attempted": "threePointersAttempted",
    "free_throws_made": "freeThrowsMade",
    "free_throws_attempted": "freeThrowsAttempted",
    "plus_minus": "plusMinusPoints",
}

In [51]:
# Aggregate player statistics over a requested time scope.
def calculate_scope_features(games, transaction_date, prefix):
    """
    Aggregate a selected collection of player-game rows into
    point-in-time features.

    The function can be used for season-to-date, career-to-date,
    or recent-game subsets by changing the prefix.
    """

    games = games.loc[games["appeared"]].copy()

    if games.empty:
        return {}

    games = games.sort_values(["game_Date", "gameId"])
    transaction_date = pd.Timestamp(transaction_date).normalize()

    output = {}

    # ---------------------------------------------------------
    # Availability and role
    # ---------------------------------------------------------

    games_played = games["gameId"].nunique()
    games_started = int(games["started"].sum())
    minutes = games["minutesPlayed"].sum(min_count=1)
    last_game_date = games["game_Date"].max()

    output.update(
        {
            f"{prefix}games_played": games_played,
            f"{prefix}games_started": games_started,
            f"{prefix}minutes": minutes,
            f"{prefix}minutes_per_game": (minutes / games_played if games_played > 0 else np.nan),
            f"{prefix}start_percentage": (100 * games_started / games_played if games_played > 0 else np.nan),
            f"{prefix}win_percentage_when_appearing": games["win"].mean(),
            f"{prefix}minutes_standard_deviation": games["minutesPlayed"].std(ddof=1),
            f"{prefix}last_game_date": last_game_date,
            f"{prefix}days_since_last_game": (transaction_date - last_game_date).days,
            f"{prefix}most_recent_team_id": games.iloc[-1]["playerteamId"],
        }
    )

    # ---------------------------------------------------------
    # Counting statistics
    # ---------------------------------------------------------

    totals = {}

    for feature_name, column_name in counting_stat_columns.items():
        total = games[column_name].sum(min_count=1)
        totals[feature_name] = total

        output[f"{prefix}{feature_name}"] = total

        output[f"{prefix}{feature_name}_per_game"] = total / games_played if pd.notna(total) and games_played > 0 else np.nan

        output[f"{prefix}{feature_name}_per_36"] = 36 * total / minutes if pd.notna(total) and pd.notna(minutes) and minutes > 0 else np.nan

    # ---------------------------------------------------------
    # Per-100-possession statistics
    # ---------------------------------------------------------

    estimated_player_possessions = games["estimatedPlayerPossessions"].sum(min_count=1)

    output[f"{prefix}estimated_player_possessions"] = estimated_player_possessions

    for feature_name, total in totals.items():
        output[f"{prefix}{feature_name}_per_100"] = (
            100 * total / estimated_player_possessions
            if pd.notna(total) and pd.notna(estimated_player_possessions) and estimated_player_possessions > 0
            else np.nan
        )

    # ---------------------------------------------------------
    # Shooting efficiency and shot profile
    # ---------------------------------------------------------

    two_pointers_made = games["twoPointersMade"].sum(min_count=1)
    two_pointers_attempted = games["twoPointersAttempted"].sum(min_count=1)

    field_goals_made = totals["field_goals_made"]
    field_goal_attempts = totals["field_goals_attempted"]
    three_pointers_made = totals["three_pointers_made"]
    three_point_attempts = totals["three_pointers_attempted"]
    free_throws_made = totals["free_throws_made"]
    free_throw_attempts = totals["free_throws_attempted"]

    output.update(
        {
            f"{prefix}field_goal_percentage": (
                field_goals_made / field_goal_attempts if pd.notna(field_goal_attempts) and field_goal_attempts > 0 else np.nan
            ),
            f"{prefix}two_point_percentage": (
                two_pointers_made / two_pointers_attempted if pd.notna(two_pointers_attempted) and two_pointers_attempted > 0 else np.nan
            ),
            f"{prefix}three_point_percentage": (
                three_pointers_made / three_point_attempts if pd.notna(three_point_attempts) and three_point_attempts > 0 else np.nan
            ),
            f"{prefix}free_throw_percentage": (
                free_throws_made / free_throw_attempts if pd.notna(free_throw_attempts) and free_throw_attempts > 0 else np.nan
            ),
            f"{prefix}effective_field_goal_percentage": (
                (field_goals_made + 0.5 * three_pointers_made) / field_goal_attempts
                if pd.notna(field_goal_attempts) and field_goal_attempts > 0
                else np.nan
            ),
            f"{prefix}three_point_attempt_rate": (
                three_point_attempts / field_goal_attempts if pd.notna(field_goal_attempts) and field_goal_attempts > 0 else np.nan
            ),
            f"{prefix}free_throw_rate": (
                free_throw_attempts / field_goal_attempts if pd.notna(field_goal_attempts) and field_goal_attempts > 0 else np.nan
            ),
            f"{prefix}assist_turnover_ratio": (
                totals["assists"] / totals["turnovers"] if pd.notna(totals["turnovers"]) and totals["turnovers"] > 0 else np.nan
            ),
        }
    )

    true_shooting_denominator = 2 * (field_goal_attempts + 0.44 * free_throw_attempts)

    output[f"{prefix}true_shooting_percentage"] = (
        totals["points"] / true_shooting_denominator if pd.notna(true_shooting_denominator) and true_shooting_denominator > 0 else np.nan
    )

    # ---------------------------------------------------------
    # Team and opponent totals
    # ---------------------------------------------------------

    team_minutes = games["team_minutes"].sum(min_count=1)
    team_game_minutes = team_minutes / 5 if pd.notna(team_minutes) else np.nan

    team_fga = games["team_fga"].sum(min_count=1)
    team_fgm = games["team_fgm"].sum(min_count=1)
    team_fta = games["team_fta"].sum(min_count=1)
    team_tov = games["team_tov"].sum(min_count=1)
    team_orb = games["team_orb"].sum(min_count=1)
    team_drb = games["team_drb"].sum(min_count=1)
    team_trb = games["team_trb"].sum(min_count=1)

    opp_fga = games["opp_fga"].sum(min_count=1)
    opp_fg3a = games["opp_fg3a"].sum(min_count=1)
    opp_orb = games["opp_orb"].sum(min_count=1)
    opp_drb = games["opp_drb"].sum(min_count=1)
    opp_trb = games["opp_trb"].sum(min_count=1)

    opponent_possessions = games["estimated_possessions"].sum(min_count=1)

    output[f"{prefix}court_time_percentage"] = (
        100 * minutes / team_game_minutes if pd.notna(minutes) and pd.notna(team_game_minutes) and team_game_minutes > 0 else np.nan
    )

    # ---------------------------------------------------------
    # Usage percentage
    # ---------------------------------------------------------

    usage_denominator = minutes * (team_fga + 0.44 * team_fta + team_tov)

    output[f"{prefix}usage_percentage"] = (
        100 * (field_goal_attempts + 0.44 * free_throw_attempts + totals["turnovers"]) * team_game_minutes / usage_denominator
        if pd.notna(usage_denominator) and usage_denominator > 0
        else np.nan
    )

    # ---------------------------------------------------------
    # Assist and turnover percentages
    # ---------------------------------------------------------

    estimated_teammate_field_goals = (
        (minutes / team_game_minutes) * team_fgm if pd.notna(minutes) and pd.notna(team_game_minutes) and team_game_minutes > 0 else np.nan
    )

    assist_denominator = estimated_teammate_field_goals - field_goals_made if pd.notna(estimated_teammate_field_goals) else np.nan

    output[f"{prefix}assist_percentage"] = (
        100 * totals["assists"] / assist_denominator if pd.notna(assist_denominator) and assist_denominator > 0 else np.nan
    )

    turnover_denominator = field_goal_attempts + 0.44 * free_throw_attempts + totals["turnovers"]

    output[f"{prefix}turnover_percentage"] = (
        100 * totals["turnovers"] / turnover_denominator if pd.notna(turnover_denominator) and turnover_denominator > 0 else np.nan
    )

    # ---------------------------------------------------------
    # Rebounding percentages
    # ---------------------------------------------------------

    offensive_rebound_denominator = minutes * (team_orb + opp_drb)
    defensive_rebound_denominator = minutes * (team_drb + opp_orb)
    total_rebound_denominator = minutes * (team_trb + opp_trb)

    output[f"{prefix}offensive_rebound_percentage"] = (
        100 * totals["offensive_rebounds"] * team_game_minutes / offensive_rebound_denominator
        if pd.notna(offensive_rebound_denominator) and offensive_rebound_denominator > 0
        else np.nan
    )

    output[f"{prefix}defensive_rebound_percentage"] = (
        100 * totals["defensive_rebounds"] * team_game_minutes / defensive_rebound_denominator
        if pd.notna(defensive_rebound_denominator) and defensive_rebound_denominator > 0
        else np.nan
    )

    output[f"{prefix}total_rebound_percentage"] = (
        100 * totals["total_rebounds"] * team_game_minutes / total_rebound_denominator
        if pd.notna(total_rebound_denominator) and total_rebound_denominator > 0
        else np.nan
    )

    # ---------------------------------------------------------
    # Steal and block percentages
    # ---------------------------------------------------------

    steal_denominator = minutes * opponent_possessions
    opponent_two_point_attempts = opp_fga - opp_fg3a
    block_denominator = minutes * opponent_two_point_attempts

    output[f"{prefix}steal_percentage"] = (
        100 * totals["steals"] * team_game_minutes / steal_denominator if pd.notna(steal_denominator) and steal_denominator > 0 else np.nan
    )

    output[f"{prefix}block_percentage"] = (
        100 * totals["blocks"] * team_game_minutes / block_denominator if pd.notna(block_denominator) and block_denominator > 0 else np.nan
    )

    # ---------------------------------------------------------
    # Game Score and plus/minus variability
    # ---------------------------------------------------------

    game_scores = games["gameScore"].dropna()

    if not game_scores.empty:
        game_score_total = game_scores.sum()

        output.update(
            {
                f"{prefix}game_score_total": game_score_total,
                f"{prefix}game_score_average": game_scores.mean(),
                f"{prefix}game_score_median": game_scores.median(),
                f"{prefix}game_score_standard_deviation": game_scores.std(ddof=1),
                f"{prefix}game_score_per_36": (36 * game_score_total / minutes if pd.notna(minutes) and minutes > 0 else np.nan),
            }
        )

    output[f"{prefix}plus_minus_standard_deviation"] = games["plusMinusPoints"].std(ddof=1)

    return output

In [52]:
trend_metrics = [
    "minutes_per_game",
    "points_per_100",
    "assists_per_100",
    "total_rebounds_per_100",
    "true_shooting_percentage",
    "effective_field_goal_percentage",
    "usage_percentage",
    "assist_percentage",
    "turnover_percentage",
    "game_score_average",
    "plus_minus_per_100",
]

In [53]:
# Build current-season, prior-season, and career features for one matched player.
def calculate_player_point_in_time_features(transaction_date, transaction_season, match_result, enriched_box_scores, team_game_context):
    """
    Use the player-matching result to calculate season-to-date,
    career-to-date, last-10, prior-team, and league-relative
    statistics as of the transaction date.
    """

    transaction_date = pd.Timestamp(transaction_date).normalize()
    player_id = match_result.get("player_id")

    transaction_date = pd.Timestamp(transaction_date).normalize()

    transaction_season = str(transaction_season)

    output = {
        "snapshot_date": transaction_date,
        "player_id": player_id,
        "player_match_status": match_result.get("match_status"),
        "box_score_player_name": match_result.get("box_score_player_name"),
        "provided_most_recent_season": match_result.get("most_recent_season"),
        "model_value": match_result.get("model_value"),
    }

    # ---------------------------------------------------------
    # Confirm that a player was matched
    # ---------------------------------------------------------

    if player_id is None or pd.isna(player_id):
        output["feature_status"] = "player_id_missing"
        return output

    player_id = int(player_id)

    # Use only appearances occurring before the transaction.
    prior_games = (
        enriched_box_scores.loc[
            (enriched_box_scores["personId"] == player_id)
            & enriched_box_scores["appeared"]
            & (enriched_box_scores["game_Date"] < transaction_date)
        ]
        .sort_values(["game_Date", "gameId"])
        .copy()
    )

    if prior_games.empty:
        output["feature_status"] = "no_prior_box_score_appearance"
        return output

    # ---------------------------------------------------------
    # Determine which season represents the player snapshot
    # ---------------------------------------------------------

    available_seasons = set(prior_games["Season"].dropna())

    latest_prior_player_season = prior_games.iloc[-1]["Season"]

    snapshot_season = transaction_season

    output.update(
        {
            "snapshot_season": snapshot_season,
            "latest_prior_player_season": (latest_prior_player_season),
            "has_current_season_appearance": (snapshot_season in available_seasons),
        }
    )

    season_games = prior_games.loc[prior_games["Season"] == snapshot_season].copy()

    last_10_games = prior_games.tail(10).copy()

    # ---------------------------------------------------------
    # Calculate season, career, and recent features
    # ---------------------------------------------------------

    output.update(calculate_scope_features(games=season_games, transaction_date=transaction_date, prefix="season_"))

    output.update(calculate_scope_features(games=prior_games, transaction_date=transaction_date, prefix="career_"))

    output.update(calculate_scope_features(games=last_10_games, transaction_date=transaction_date, prefix="last10_"))

    output.update({"career_seasons_played": prior_games["Season"].nunique(), "career_first_game_date": prior_games["game_Date"].min()})

    # ---------------------------------------------------------
    # Identify the player's team immediately before the trade
    # ---------------------------------------------------------

    current_team_id = prior_games.iloc[-1]["playerteamId"]

    output["most_recent_player_team_id"] = current_team_id

    # ---------------------------------------------------------
    # Prior-team performance before the transaction
    # ---------------------------------------------------------

    prior_team_games = team_game_context.loc[
        (team_game_context["playerteamId"] == current_team_id)
        & (team_game_context["Season"] == snapshot_season)
        & (team_game_context["game_Date"] < transaction_date)
    ].copy()

    if not prior_team_games.empty:
        team_games_played = prior_team_games["gameId"].nunique()
        team_wins = prior_team_games["win"].sum()
        team_losses = team_games_played - team_wins

        team_points = prior_team_games["team_points"].sum()
        opponent_points = prior_team_games["opp_points"].sum()
        team_possessions = prior_team_games["estimated_possessions"].sum()

        prior_team_ortg = 100 * team_points / team_possessions if team_possessions > 0 else np.nan

        prior_team_drtg = 100 * opponent_points / team_possessions if team_possessions > 0 else np.nan

        output.update(
            {
                "prior_team_id": current_team_id,
                "prior_team_games": team_games_played,
                "prior_team_wins": team_wins,
                "prior_team_losses": team_losses,
                "prior_team_win_percentage": (team_wins / team_games_played if team_games_played > 0 else np.nan),
                "prior_team_points_per_game": (team_points / team_games_played if team_games_played > 0 else np.nan),
                "prior_team_opponent_points_per_game": (opponent_points / team_games_played if team_games_played > 0 else np.nan),
                "prior_team_point_differential_per_game": (
                    (team_points - opponent_points) / team_games_played if team_games_played > 0 else np.nan
                ),
                "prior_team_offensive_rating": prior_team_ortg,
                "prior_team_defensive_rating": prior_team_drtg,
                "prior_team_net_rating": prior_team_ortg - prior_team_drtg,
                "prior_team_pace": prior_team_games["pace"].mean(),
            }
        )

    # ---------------------------------------------------------
    # Player availability during the current team stint
    # ---------------------------------------------------------

    player_stint_games = season_games.loc[season_games["playerteamId"] == current_team_id].copy()

    if not player_stint_games.empty and not prior_team_games.empty:
        stint_start_date = player_stint_games["game_Date"].min()

        team_stint_games = prior_team_games.loc[prior_team_games["game_Date"] >= stint_start_date].copy()

        player_stint_game_count = player_stint_games["gameId"].nunique()
        team_stint_game_count = team_stint_games["gameId"].nunique()
        player_stint_minutes = player_stint_games["minutesPlayed"].sum()
        available_stint_minutes = team_stint_games["game_minutes"].sum()

        output.update(
            {
                "stint_start_date": stint_start_date,
                "stint_player_games": player_stint_game_count,
                "stint_team_games": team_stint_game_count,
                "stint_games_played_percentage": (
                    100 * player_stint_game_count / team_stint_game_count if team_stint_game_count > 0 else np.nan
                ),
                "stint_minutes": player_stint_minutes,
                "stint_court_time_percentage": (
                    100 * player_stint_minutes / available_stint_minutes if available_stint_minutes > 0 else np.nan
                ),
            }
        )

    # ---------------------------------------------------------
    # League shooting environment before the transaction
    # ---------------------------------------------------------

    league_games = team_game_context.loc[
        (team_game_context["Season"] == snapshot_season) & (team_game_context["game_Date"] < transaction_date)
    ].copy()

    if not league_games.empty:
        league_points = league_games["team_points"].sum()
        league_fgm = league_games["team_fgm"].sum()
        league_fga = league_games["team_fga"].sum()
        league_three_pm = league_games["team_fg3m"].sum()
        league_fta = league_games["team_fta"].sum()

        league_ts_denominator = 2 * (league_fga + 0.44 * league_fta)

        league_ts = league_points / league_ts_denominator if league_ts_denominator > 0 else np.nan

        league_efg = (league_fgm + 0.5 * league_three_pm) / league_fga if league_fga > 0 else np.nan

        season_ts = output.get("season_true_shooting_percentage")
        season_efg = output.get("season_effective_field_goal_percentage")

        output.update(
            {
                "league_true_shooting_percentage": league_ts,
                "league_effective_field_goal_percentage": league_efg,
                "season_true_shooting_plus": (
                    100 * season_ts / league_ts if pd.notna(season_ts) and pd.notna(league_ts) and league_ts > 0 else np.nan
                ),
                "season_effective_field_goal_plus": (
                    100 * season_efg / league_efg if pd.notna(season_efg) and pd.notna(league_efg) and league_efg > 0 else np.nan
                ),
            }
        )

    # ---------------------------------------------------------
    # Current-season performance relative to career performance
    # ---------------------------------------------------------

    for metric in trend_metrics:
        season_value = output.get(f"season_{metric}")
        career_value = output.get(f"career_{metric}")

        output[f"season_minus_career_{metric}"] = (
            season_value - career_value if pd.notna(season_value) and pd.notna(career_value) else np.nan
        )

    output["feature_status"] = "calculated"

    return output

In [54]:
# Keep only transactions with a supported feature season.
supported_seasons = set(enriched_Box_Scores["Season"].dropna().unique())

transactions_In_Scope = transactions.loc[transactions["feature_season"].isin(supported_seasons)].copy().reset_index(drop=True)

In [55]:
print("Original transaction rows:", len(transactions))

print("In-scope transaction rows:", len(transactions_In_Scope))

print("Removed rows:", len(transactions) - len(transactions_In_Scope))

print("Earliest supported transaction season:", transactions_In_Scope["Season"].min())

print("Unsupported seasons remaining:", (~transactions_In_Scope["Season"].isin(supported_seasons)).sum())

Original transaction rows: 2749
In-scope transaction rows: 2749
Removed rows: 0
Earliest supported transaction season: 1985-86
Unsupported seasons remaining: 1808


In [56]:
# Resolve players and calculate point-in-time features for every transaction side.
transaction_player_feature_records = []
transaction_feature_audit_records = []

for row_number, (row_index, transaction_row) in enumerate(transactions_In_Scope.iterrows(), start=1):
    # Return matching details needed by the feature audit.
    match_result = match_transaction_players(transaction_row, include_details=True)

    acquired_matches = match_result.get("Acquired", {})
    relinquished_matches = match_result.get("Relinquished", {})

    calculated_player_count = 0
    failed_player_count = 0

    for transaction_side, side_matches in [("Acquired", acquired_matches), ("Relinquished", relinquished_matches)]:
        for transaction_player_name, player_match in side_matches.items():

            player_features = calculate_player_point_in_time_features(
                transaction_date=transaction_row["Date"],
                transaction_season=transaction_row["feature_season"],
                match_result=player_match,
                enriched_box_scores=enriched_Box_Scores,
                team_game_context=team_Games,
            )

            feature_status = player_features.get("feature_status", "unknown")

            if feature_status == "calculated":
                calculated_player_count += 1
            else:
                failed_player_count += 1

            feature_record = {
                "transaction_row_id": row_index,
                "source_transaction_index": transaction_row.get("index", row_index),
                "transaction_date": transaction_row["Date"],
                "transaction_season": transaction_row["comparison_season"],
                "feature_season": transaction_row["feature_season"],
                "season_segment": transaction_row["season_segment"],
                "transaction_team": transaction_row["Team"],
                "transaction_team_id": transaction_row["teamID"],
                "transaction_side": transaction_side,
                "transaction_player_name": transaction_player_name,
                "transaction_notes": transaction_row.get("Notes"),
                "source_current_wins": transaction_row.get("current_Wins"),
                "source_current_losses": transaction_row.get("current_Losses"),
                "source_games_played": transaction_row.get("games_Played"),
                "source_remaining_wins": transaction_row.get("remaining_Wins"),
                "source_remaining_losses": transaction_row.get("remaining_Losses"),
                **player_features,
            }

            transaction_player_feature_records.append(feature_record)

    transaction_feature_audit_records.append(
        {
            "transaction_row_id": row_index,
            "transaction_date": transaction_row["Date"],
            "transaction_season": transaction_row["comparison_season"],
            "feature_season": transaction_row["feature_season"],
            "season_segment": transaction_row["season_segment"],
            "transaction_team": transaction_row["Team"],
            "acquired_match_count": len(acquired_matches),
            "relinquished_match_count": len(relinquished_matches),
            "total_match_count": (len(acquired_matches) + len(relinquished_matches)),
            "calculated_player_count": calculated_player_count,
            "failed_player_count": failed_player_count,
            "has_any_match": bool(acquired_matches or relinquished_matches),
        }
    )

    if row_number % 100 == 0:
        print(f"Processed {row_number:,} " f"of {len(transactions_In_Scope):,} transactions")

Processed 100 of 2,749 transactions


Processed 200 of 2,749 transactions


Processed 300 of 2,749 transactions


Processed 400 of 2,749 transactions


Processed 500 of 2,749 transactions


...

Processed 2,300 of 2,749 transactions


Processed 2,400 of 2,749 transactions


Processed 2,500 of 2,749 transactions


Processed 2,600 of 2,749 transactions


Processed 2,700 of 2,749 transactions


In [57]:
# Create player-feature and transaction-audit tables.
transaction_Player_Features = pd.DataFrame(transaction_player_feature_records)

transaction_Feature_Audit = pd.DataFrame(transaction_feature_audit_records)

In [58]:
print("In-scope transactions:", len(transactions_In_Scope))

print("Transaction audit rows:", len(transaction_Feature_Audit))

print("Matched player-feature rows:", len(transaction_Player_Features))

In-scope transactions: 2749
Transaction audit rows: 2749
Matched player-feature rows: 6468


In [59]:
transaction_Feature_Audit["has_any_match"].value_counts(dropna=False)

has_any_match
True     2653
False      96
Name: count, dtype: int64

In [60]:
transaction_Feature_Audit[
    ["acquired_match_count", "relinquished_match_count", "total_match_count", "calculated_player_count", "failed_player_count"]
].sum()

acquired_match_count        3238
relinquished_match_count    3230
total_match_count           6468
calculated_player_count     5194
failed_player_count         1274
dtype: int64

In [61]:
transaction_Feature_Audit.groupby("transaction_season").agg(
    transaction_rows=("transaction_row_id", "size"),
    transactions_with_matches=("has_any_match", "sum"),
    matched_players=("total_match_count", "sum"),
    calculated_players=("calculated_player_count", "sum"),
    failed_players=("failed_player_count", "sum"),
).assign(transaction_match_percentage=lambda df: (100 * df["transactions_with_matches"] / df["transaction_rows"]))

,transaction_rows,transactions_with_matches,matched_players,calculated_players,failed_players,transaction_match_percentage
transaction_season,,,,,,
1985-86,14,12,22,22,0,85.714286
1986-87,84,70,124,108,16,83.333333
1987-88,77,59,120,110,10,76.623377
1988-89,65,55,116,101,15,84.615385
1989-90,70,64,110,94,16,91.428571
1990-91,74,68,120,104,16,91.891892
1991-92,51,51,88,68,20,100.000000
1992-93,58,56,120,100,20,96.551724
1993-94,56,54,114,96,18,96.428571


In [62]:
transaction_Player_Features["feature_status"].value_counts(dropna=False)

feature_status
calculated                       5194
no_prior_box_score_appearance     710
player_id_missing                 564
Name: count, dtype: int64

In [63]:
failed_Player_Features = transaction_Player_Features.loc[transaction_Player_Features["feature_status"].ne("calculated")].copy()

failed_Player_Features[
    [
        "transaction_row_id",
        "transaction_date",
        "transaction_season",
        "transaction_team",
        "transaction_side",
        "transaction_player_name",
        "player_id",
        "player_match_status",
        "feature_status",
    ]
].head(30)

,transaction_row_id,transaction_date,transaction_season,transaction_team,transaction_side,transaction_player_name,player_id,player_match_status,feature_status
32,18,1986-06-17,1986-87,Trail Blazers,Acquired,Larry Krystkowiak,1474.0,exact_match,no_prior_box_score_appearance
35,19,1986-06-17,1986-87,Bulls,Relinquished,Larry Krystkowiak,1474.0,exact_match,no_prior_box_score_appearance
36,20,1986-06-17,1986-87,Cavaliers,Acquired,Mark Price,899.0,exact_match,no_prior_box_score_appearance
38,21,1986-06-17,1986-87,Hawks,Acquired,Ken Barlow,NaN,unresolved_name,player_id_missing
39,21,1986-06-17,1986-87,Hawks,Relinquished,Ron Kellogg,NaN,unresolved_name,player_id_missing
40,21,1986-06-17,1986-87,Hawks,Relinquished,Billy Thompson,1631.0,exact_match,no_prior_box_score_appearance
41,22,1986-06-17,1986-87,Lakers,Acquired,Ron Kellogg,NaN,unresolved_name,player_id_missing
42,22,1986-06-17,1986-87,Lakers,Acquired,Billy Thompson,1631.0,exact_match,no_prior_box_score_appearance
44,22,1986-06-17,1986-87,Lakers,Relinquished,Ken Barlow,NaN,unresolved_name,player_id_missing
45,23,1986-06-17,1986-87,Mavericks,Relinquished,Mark Price,899.0,exact_match,no_prior_box_score_appearance


In [64]:
duplicate_player_transactions = transaction_Player_Features.duplicated(
    subset=["transaction_row_id", "transaction_side", "player_id"], keep=False
)

print("Duplicate player-transaction rows:", duplicate_player_transactions.sum())

Duplicate player-transaction rows: 40


In [65]:
transaction_Player_Features.loc[
    duplicate_player_transactions,
    ["transaction_row_id", "transaction_date", "transaction_team", "transaction_side", "transaction_player_name", "player_id"],
].sort_values(["transaction_row_id", "transaction_side", "player_id"])

,transaction_row_id,transaction_date,transaction_team,transaction_side,transaction_player_name,player_id
2960,1304,2005-08-02,Celtics,Acquired,Curtis Borchardt,NaN
2961,1304,2005-08-02,Celtics,Acquired,Albert Miralles,NaN
2971,1306,2005-08-02,Heat,Acquired,Andre Emmett,NaN
2973,1306,2005-08-02,Heat,Acquired,Roberto Duenas,NaN
3501,1517,2008-02-21,Grizzlies,Acquired,Marcus Vinicius Viera de Souza,NaN
3502,1517,2008-02-21,Grizzlies,Acquired,Malick Badiane,NaN
4420,1893,2011-06-23,Pacers,Relinquished,Erazem Lorbek,NaN
4422,1893,2011-06-23,Pacers,Relinquished,Davis Bertans,NaN
4428,1895,2011-06-23,Spurs,Acquired,Erazem Lorbek,NaN
4430,1895,2011-06-23,Spurs,Acquired,Davis Bertans,NaN


In [66]:
calculated_Player_Features = transaction_Player_Features.loc[transaction_Player_Features["feature_status"].eq("calculated")]

season_mismatch_rows = calculated_Player_Features.loc[
    calculated_Player_Features["snapshot_season"].ne(calculated_Player_Features["transaction_season"])
]

print("Snapshot-season mismatches:", len(season_mismatch_rows))

Snapshot-season mismatches: 2763


In [67]:
unmatched_transaction_ids = transaction_Feature_Audit.loc[~transaction_Feature_Audit["has_any_match"], "transaction_row_id"]

transactions_In_Scope.loc[unmatched_transaction_ids, ["Date", "Team", "Acquired", "Relinquished", "Notes"]]

,Date,Team,Acquired,Relinquished,Notes
0,1985-10-29,Bulls,Spurs agreed to not exercise their right of fi...,cash,trade with Spurs
1,1985-10-29,Spurs,cash,Spurs agreed to not exercise their right of fi...,trade with Bulls
30,1986-08-05,Trail Blazers,Bulls agreed to not exercise their right of fi...,1992 second round pick (#52-Matt Steigenga),trade with Bulls
31,1986-08-05,Bulls,1992 second round pick (#52-Matt Steigenga),Bulls agreed to not exercise their right of fi...,trade with Blazers
56,1986-10-02,Bulls,"1989 first round pick (#6-Stacey King), 1988 s...",Bulls agreed to not exercise their right of fi...,trade with Nets
...,...,...,...,...,...
1950,2012-03-15,Warriors,2012 second round pick (less favorable of Hawk...,cash considerations,trade with Hawks
2473,2017-01-06,Trail Blazers,2017 first round pick (#26-Caleb Swanigan),2018 first round pick (#25-Moritz Wagner / Moe...,trade with Cavaliers
2474,2017-01-06,Cavaliers,2018 first round pick (#25-Moritz Wagner / Moe...,2017 first round pick (#26-Caleb Swanigan),trade with Blazers
2509,2017-06-19,76ers,2017 first round pick (#1-Markelle Fultz),"2017 first round pick (#3-Jayson Tatum), first...",trade with Celtics


In [68]:
# Save point-in-time features and audit outputs.
transaction_Player_Features.to_parquet("../data/interim/transaction_player_features_offseason.parquet")
transactions_In_Scope.to_parquet("../data/interim/transactions_in_scope_offseason.parquet", index=True)
transaction_Feature_Audit.to_parquet("../data/interim/transaction_Feature_Audit_offseason.parquet", index=True)